# Task 2 — Build Time Series Forecasting Models
**GMF Investments | Time Series Forecasting for Portfolio Management Optimization**

Objective: build, train, and compare an ARIMA/SARIMA model and an LSTM model to
forecast TSLA's closing price, using a chronological train/test split
(train 2015–2024, test 2025–2026) as required — random shuffling is invalid for
time series data.


In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

from src.forecasting import (
    chronological_split,
    evaluate_forecast,
    fit_auto_arima,
    forecast_arima,
    create_sequences,
    build_lstm_model,
    iterative_lstm_forecast,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

SPLIT_DATE = "2025-01-01"


## 1. Load Cleaned Data & Prepare Train/Test Split

We use the TSLA adjusted close price series produced in Task 1
(`data/processed/adj_close_prices.csv`). The split is chronological:
train on data before `2025-01-01`, test on everything from that date onward.
This mirrors a realistic forecasting setup where the model only ever sees
the past.


In [ ]:
prices = pd.read_csv("../data/processed/adj_close_prices.csv", index_col=0, parse_dates=True)
tsla = prices["TSLA"]

train, test = chronological_split(tsla, split_date=SPLIT_DATE)
print(f"Train: {train.index.min().date()} -> {train.index.max().date()}  ({len(train)} rows)")
print(f"Test:  {test.index.min().date()} -> {test.index.max().date()}  ({len(test)} rows)")

fig, ax = plt.subplots()
ax.plot(train.index, train.values, label="Train")
ax.plot(test.index, test.values, label="Test")
ax.axvline(pd.Timestamp(SPLIT_DATE), color="black", linestyle="--", linewidth=1)
ax.set_title("TSLA Adjusted Close — Train/Test Split")
ax.legend()
plt.tight_layout()
plt.show()


## 2. ARIMA / SARIMA Model

`auto_arima` searches over (p, d, q) — and (P, D, Q, m) if `seasonal=True` —
to minimize AIC, avoiding a manual ACF/PACF-driven grid search. We fit on
price levels directly; `auto_arima` handles the differencing order (`d`)
internally based on the same style of stationarity test used in Task 1.


In [ ]:
arima_model = fit_auto_arima(
    train,
    seasonal=False,   # start with non-seasonal ARIMA; daily stock prices
                      # rarely show strong fixed-period seasonality
    max_p=5, max_q=5, max_d=2,
    trace=True,
)
print(arima_model.summary())


In [ ]:
n_test = len(test)
arima_forecast, arima_conf_int = forecast_arima(arima_model, n_periods=n_test, alpha=0.05)

arima_forecast_series = pd.Series(arima_forecast, index=test.index)
arima_lower = pd.Series(arima_conf_int[:, 0], index=test.index)
arima_upper = pd.Series(arima_conf_int[:, 1], index=test.index)

fig, ax = plt.subplots()
ax.plot(train.index[-200:], train.values[-200:], label="Train (last 200d)")
ax.plot(test.index, test.values, label="Actual (test)")
ax.plot(test.index, arima_forecast_series, label="ARIMA Forecast", linestyle="--")
ax.fill_between(test.index, arima_lower, arima_upper, alpha=0.2, label="95% CI")
ax.set_title("ARIMA — Test Period Forecast vs. Actual")
ax.legend()
plt.tight_layout()
plt.show()

arima_metrics = evaluate_forecast(test.values, arima_forecast_series.values, model_name="ARIMA")
arima_metrics


### Optional: SARIMA

If a seasonal pattern is suspected (e.g. a ~5-day trading-week effect), refit
with `seasonal=True` and an appropriate `m`. This is commented out by default
since daily equity prices typically don't show strong fixed seasonality —
uncomment to compare.


In [ ]:
# sarima_model = fit_auto_arima(train, seasonal=True, m=5, max_p=3, max_q=3, max_P=2, max_Q=2, trace=True)
# print(sarima_model.summary())


## 3. LSTM Model

### 3.1 Scale data and build sequences

LSTMs are sensitive to input scale, so we fit a `MinMaxScaler` on the
**training data only** (to avoid leaking test-period information) and use a
60-day lookback window: the last 60 days of prices predict the next day.


In [ ]:
WINDOW = 60

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train.values.reshape(-1, 1)).flatten()

# To forecast the test period we need the last WINDOW days of train immediately
# preceding it as context, so scale train+test together using the *train-fit* scaler
# (not refitting on test) and build sequences across the boundary.
full_scaled = scaler.transform(tsla.values.reshape(-1, 1)).flatten()

X_train, y_train = create_sequences(train_scaled, window=WINDOW)
print("X_train shape:", X_train.shape, " y_train shape:", y_train.shape)


### 3.2 Build and train the model

In [ ]:
lstm_model = build_lstm_model(window=WINDOW, units=(50, 50), dropout=0.2)
lstm_model.summary()


In [ ]:
history = lstm_model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_split=0.1,
    verbose=1,
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history["loss"], label="Train Loss")
ax.plot(history.history["val_loss"], label="Val Loss")
ax.set_title("LSTM Training Loss (MSE, scaled)")
ax.set_xlabel("Epoch")
ax.legend()
plt.tight_layout()
plt.show()


### 3.3 Forecast the test period

We seed the forecast with the last `WINDOW` scaled training values, then
iteratively predict one step ahead, feeding each prediction back in as
input for the next step (as required for a multi-step LSTM forecast).


In [ ]:
last_window = train_scaled[-WINDOW:]
lstm_scaled_forecast = iterative_lstm_forecast(lstm_model, last_window, n_periods=len(test))

lstm_forecast = scaler.inverse_transform(lstm_scaled_forecast.reshape(-1, 1)).flatten()
lstm_forecast_series = pd.Series(lstm_forecast, index=test.index)

fig, ax = plt.subplots()
ax.plot(train.index[-200:], train.values[-200:], label="Train (last 200d)")
ax.plot(test.index, test.values, label="Actual (test)")
ax.plot(test.index, lstm_forecast_series, label="LSTM Forecast", linestyle="--")
ax.set_title("LSTM — Test Period Forecast vs. Actual")
ax.legend()
plt.tight_layout()
plt.show()

lstm_metrics = evaluate_forecast(test.values, lstm_forecast_series.values, model_name="LSTM")
lstm_metrics


## 4. Model Comparison

Multi-step iterative forecasts (both ARIMA's `n_periods`-ahead and the LSTM's
feed-forward loop) tend to drift over long horizons since errors compound —
expect both models' accuracy to degrade further into the ~18-month test
window. This is exactly why Task 3 leans on confidence intervals rather than
treating the point forecast as certain.


In [ ]:
comparison = pd.DataFrame([arima_metrics, lstm_metrics]).set_index("model")
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test.index, test.values, label="Actual", color="black", linewidth=1.5)
ax.plot(test.index, arima_forecast_series, label="ARIMA", linestyle="--")
ax.plot(test.index, lstm_forecast_series, label="LSTM", linestyle="--")
ax.set_title("TSLA Test-Period Forecast Comparison")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Discussion — Which Model Performed Better, and Why?

*(Fill in once run against live results — talking points below.)*

- **ARIMA/SARIMA** is a linear model on (differenced) price levels. It tends
  to perform well for short horizons and produces principled confidence
  intervals, but its point forecasts typically flatten out or drift linearly
  over long horizons since it has no mechanism to capture nonlinear regime
  changes.
- **LSTM** can in principle capture nonlinear patterns and longer-range
  dependencies, but with an iterative multi-step forecast, small early errors
  compound over an 18-month horizon — watch for the forecast decaying toward
  a flat trend or diverging implausibly by the end of the test window.
- **Interpretability vs. flexibility trade-off:** ARIMA's parameters
  (p, d, q) are directly interpretable (autoregressive lag, differencing
  order, moving-average lag); the LSTM is a black box by comparison, which
  matters for a client-facing advisory context like GMF's.
- Compare `MAE`/`RMSE`/`MAPE` directly: the model with lower error wins
  numerically, but note *where* the errors are concentrated (e.g., does one
  model do better early in the test window and worse later?) since that
  matters for how much to trust the 6–12 month forecast in Task 3.
